In [1]:
import pandas as pd
import numpy as np
import matplotlib as plt

In [2]:
!pip install mediapipe

In [3]:
import os
import cv2
import numpy as np
import mediapipe as mp
from time import time

In [4]:
mp_pose = mp.solutions.pose

AttributeError: module 'mediapipe' has no attribute 'solutions'

In [5]:
!pip install mediapipe==0.10.14

ERROR: Could not find a version that satisfies the requirement mediapipe==0.10.14 (from versions: 0.10.30, 0.10.31, 0.10.32, 0.10.33, 0.10.35, 1.0.0, 1.0.1, 1.1.0, 1.1.1, 1.2.0, 1.2.2)
ERROR: No matching distribution found for mediapipe==0.10.14


In [6]:
pip install mediapipe==0.10.14

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement mediapipe==0.10.14 (from versions: 0.10.30, 0.10.31, 0.10.32, 0.10.33, 0.10.35, 1.0.0, 1.0.1, 1.1.0, 1.1.1, 1.2.0, 1.2.2)
ERROR: No matching distribution found for mediapipe==0.10.14


In [7]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

base_options = python.BaseOptions(model_asset_path='pose_landmarker_heavy.task')
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_poses=2
)
detector = vision.PoseLandmarker.create_from_options(options)

mp_image = mp.Image.create_from_file(r"C:\Users\VIVEK\Downloads\good.png")

detection_result = detector.detect(mp_image)
num_people = len(detection_result.pose_landmarks)

print(f"Number of people detected: {num_people}")

# 4. Print the Nose coordinates (Landmark 0) for the first person
if num_people > 0:
    first_person_landmarks = detection_result.pose_landmarks[0]
    nose = first_person_landmarks[0] 
    
    print(f"Nose X: {nose.x}")
    print(f"Nose Y: {nose.y}")
    print(f"Nose Visibility: {nose.visibility}")
# Landmark 27 is the Left Ankle, 28 is the Right Ankle
    left_ankle = first_person_landmarks[27]
    right_ankle = first_person_landmarks[28]
    
    print(f"Left Ankle Visibility: {left_ankle.visibility}")
    print(f"Right Ankle Visibility: {right_ankle.visibility}")
    
    # Write your first FitFlick rule:
    if left_ankle.visibility < 0.5 or right_ankle.visibility < 0.5:
        print("STATUS: FAIL - Ankles are cut off!")
    else:
        print("STATUS: PASS - Full body is in the frame.")
    left_toe = first_person_landmarks[31]
    right_toe = first_person_landmarks[32]
    
    # MediaPipe Y coordinates go from 0.0 (top of image) to 1.0 (bottom of image)
    # If the toe's Y is greater than 0.98, the foot is practically touching the bottom border
    if left_toe.y > 0.98 or right_toe.y > 0.98:
        print("STATUS: FAIL - Feet are cut off at the bottom edge!")
    else:
        print("Done")

Number of people detected: 1
Nose X: 0.4949563443660736
Nose Y: 0.09959250688552856
Nose Visibility: 0.99994957447052
Left Ankle Visibility: 0.9888859987258911
Right Ankle Visibility: 0.9863109588623047
STATUS: PASS - Full body is in the frame.
Done


In [8]:
import math

def calculate_angle(a, b, c):
    """Calculates the 2D angle at joint 'b' (e.g., knee) using joints a, b, c."""
    radians = math.atan2(c.y - b.y, c.x - b.x) - math.atan2(a.y - b.y, a.x - b.x)
    angle = abs(radians * 180.0 / math.pi)
    if angle > 180.0:
        angle = 360.0 - angle
    return angle

image_path = r"C:\Users\VIVEK\Downloads\good.png"

# --- 1. OPENCV IMAGE QUALITY CHECKS ---
img = cv2.imread(image_path)

# A. Size Check
h, w, channels = img.shape
if h < 500 or w < 350:
    print(f"STATUS: FAIL - Image too small ({w}x{h})")

# B. Brightness & Blur Check
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
brightness = np.mean(gray)
blur_score = cv2.Laplacian(gray, cv2.CV_64F).var()

print(f"Image Size: {w}x{h}")
print(f"Brightness Score: {brightness:.2f}")
print(f"Blur Score: {blur_score:.2f}")

if brightness < 40:
    print("STATUS: FAIL - Image is too dark!")
elif blur_score < 80:
    print("STATUS: FAIL - Image is too blurry!")
else:
    print("STATUS: PASS - Image quality is good.")

# --- 2. MEDIAPIPE POSE DETECTION ---
# (Keep your existing MediaPipe code here, but change the file loader to use the variable)
mp_image = mp.Image.create_from_file(image_path)

base_options = python.BaseOptions(model_asset_path='pose_landmarker_heavy.task')
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_poses=2
)
detector = vision.PoseLandmarker.create_from_options(options)

mp_image = mp.Image.create_from_file(r"C:\Users\VIVEK\Downloads\good.png")

detection_result = detector.detect(mp_image)
num_people = len(detection_result.pose_landmarks)

print(f"Number of people detected: {num_people}")

# 4. Print the Nose coordinates (Landmark 0) for the first person
if num_people > 0:
    first_person_landmarks = detection_result.pose_landmarks[0]
    nose = first_person_landmarks[0] 
    
    print(f"Nose X: {nose.x}")
    print(f"Nose Y: {nose.y}")
    print(f"Nose Visibility: {nose.visibility}")
# Landmark 27 is the Left Ankle, 28 is the Right Ankle
    left_ankle = first_person_landmarks[27]
    right_ankle = first_person_landmarks[28]
    
    print(f"Left Ankle Visibility: {left_ankle.visibility}")
    print(f"Right Ankle Visibility: {right_ankle.visibility}")
    
    # Write your first FitFlick rule:
    if left_ankle.visibility < 0.5 or right_ankle.visibility < 0.5:
        print("STATUS: FAIL - Ankles are cut off!")
    else:
        print("STATUS: PASS - Full body is in the frame.")
    left_toe = first_person_landmarks[31]
    right_toe = first_person_landmarks[32]
    
    # MediaPipe Y coordinates go from 0.0 (top of image) to 1.0 (bottom of image)
    # If the toe's Y is greater than 0.98, the foot is practically touching the bottom border
    if left_toe.y > 0.98 or right_toe.y > 0.98:
        print("STATUS: FAIL - Feet are cut off at the bottom edge!")
    else:
        print("Done")

# --- 3. STANDING & FACING CAMERA CHECKS ---
    left_hip = first_person_landmarks[23]
    left_knee = first_person_landmarks[25]
    left_shoulder = first_person_landmarks[11]
    right_shoulder = first_person_landmarks[12]

    # A. Standing Check (Measure the Hip-Knee-Ankle angle)
    # A straight leg is ~180 degrees. A sitting person drops below ~140.
    knee_angle = calculate_angle(left_hip, left_knee, left_ankle)
    print(f"Left Knee Angle: {knee_angle:.1f} degrees")

    if knee_angle < 150:
        print("STATUS: FAIL - Person is sitting or bending.")
    else:
        print("STATUS: PASS - Person is standing straight.")

    # B. Facing Camera Check (Measure horizontal shoulder distance)
    # If someone stands sideways (profile view), the distance between their left 
    # and right shoulder on a 2D screen shrinks to almost 0.
    shoulder_width = abs(left_shoulder.x - right_shoulder.x)
    print(f"Shoulder Width (Normalized): {shoulder_width:.3f}")

    if shoulder_width < 0.08:
        print("STATUS: FAIL - Person is turned sideways.")
    else:
        print("STATUS: PASS - Person is facing the camera.")

Image Size: 432x653
Brightness Score: 201.49
Blur Score: 241.33
STATUS: PASS - Image quality is good.
Number of people detected: 1
Nose X: 0.4949563443660736
Nose Y: 0.09959250688552856
Nose Visibility: 0.99994957447052
Left Ankle Visibility: 0.9888859987258911
Right Ankle Visibility: 0.9863109588623047
STATUS: PASS - Full body is in the frame.
Done
Left Knee Angle: 174.8 degrees
STATUS: PASS - Person is standing straight.
Shoulder Width (Normalized): 0.262
STATUS: PASS - Person is facing the camera.


In [14]:
import cv2
import math
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# Initialize your MediaPipe detector ONCE outside the function
base_options = python.BaseOptions(model_asset_path='pose_landmarker_heavy.task')
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_poses=2
)
detector = vision.PoseLandmarker.create_from_options(options)

def calculate_angle(a, b, c):
    """Calculates the 2D angle at joint 'b' (e.g., knee) using joints a, b, c."""
    radians = math.atan2(c.y - b.y, c.x - b.x) - math.atan2(a.y - b.y, a.x - b.x)
    angle = abs(radians * 180.0 / math.pi)
    if angle > 180.0:
        angle = 360.0 - angle
    return angle

def check_photo(image_path):
    """
    Returns a dictionary: {"status": "pass" or "fail", "reason": "Explanation"}
    """
    img = cv2.imread(image_path)
    if img is None:
        return {"status": "fail", "reason": "Image not found or unreadable"}

    # 1. Calculate Size, Brightness, and Blur FIRST
    h, w, channels = img.shape
    if h < 500 or w < 220:
        return {"status": "fail", "reason": f"Image too small ({w}x{h})"}

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    brightness = np.mean(gray)
    blur_score = cv2.Laplacian(gray, cv2.CV_64F).var()

    if brightness < 20:
        return {"status": "fail", "reason": "Image is too dark"}
    if blur_score < 50:
        return {"status": "fail", "reason": "Image is too blurry"}

    # 2. Run MediaPipe only if the image quality passed
    mp_image = mp.Image.create_from_file(image_path)
    detection_result = detector.detect(mp_image)
    num_people = len(detection_result.pose_landmarks)

    if num_people == 0:
        return {"status": "fail", "reason": "No person detected"}
    if num_people > 1:
        return {"status": "fail", "reason": "Multiple people detected"}

    # 3. Geometry Checks (Since num_people is exactly 1)
    first_person_landmarks = detection_result.pose_landmarks[0]
    
    left_ankle = first_person_landmarks[27]
    right_ankle = first_person_landmarks[28]
    left_toe = first_person_landmarks[31]
    right_toe = first_person_landmarks[32]

    # Full Body & Edge Check
    if left_ankle.visibility < 0.5 or right_ankle.visibility < 0.5:
        return {"status": "fail", "reason": "Ankles are cut off"}
    if left_toe.y > 0.97 or right_toe.y > 0.98:
        return {"status": "fail", "reason": "Feet are cut off at the bottom edge"}

    # Standing Check
    left_hip = first_person_landmarks[23]
    left_knee = first_person_landmarks[25]
    knee_angle = calculate_angle(left_hip, left_knee, left_ankle)
    
    if knee_angle < 150:
        return {"status": "fail", "reason": "Person is sitting or bending"}

    # Facing Camera Check
    left_shoulder = first_person_landmarks[11]
    right_shoulder = first_person_landmarks[12]
    shoulder_width = abs(left_shoulder.x - right_shoulder.x)

    if shoulder_width < 0.08:
        return {"status": "fail", "reason": "Person is turned sideways"}

    # If it survives all the above 'if' statements, it is a perfect photo!
    return {"status": "pass", "reason": "Perfect photo for virtual try-on"}

import os

# Start by pointing to your 30 tune photos!
folder_path = r"C:\Users\VIVEK\Downloads\WhatsApp Unknown 2026-11-09 at 11.43.06 AM"

total_good = 0
good_rejected = 0

total_bad = 0
bad_accepted = 0

print(f"--- Evaluating folder: {folder_path} ---\n")

for filename in os.listdir(folder_path):
    if not filename.lower().endswith((".jpg", ".jpeg", ".png")):
        continue

    full_path = os.path.join(folder_path, filename)
    
    # 1. Determine Ground Truth from filename (or CSV if you made one)
    if filename.lower().startswith("good"):
        true_status = "pass"
        total_good += 1
    else:
        true_status = "fail"
        total_bad += 1

    # 2. Run your checker (check_photo ONLY reads pixels, never the filename!)
    prediction = check_photo(full_path)
    pred_status = prediction["status"]
    pred_reason = prediction["reason"]

    # 3. Check for mistakes and count them
    if true_status == "pass" and pred_status == "fail":
        good_rejected += 1
        print(f"[FALSE REJECTION] {filename} -> Failed because: {pred_reason}")
        
    elif true_status == "fail" and pred_status == "pass":
        bad_accepted += 1
        print(f"[FALSE ACCEPTANCE] {filename} -> Bad photo slipped through!")

# 4. Calculate the 2 FitFlick Business Metrics (Requirement 4)
false_rejection_rate = (good_rejected / total_good * 100) if total_good > 0 else 0
false_acceptance_rate = (bad_accepted / total_bad * 100) if total_bad > 0 else 0

print("\n--- FINAL SUMMARY ---")
print(f"Total Good Photos: {total_good} | Wrongly Rejected: {good_rejected} ({false_rejection_rate:.1f}%)")
print(f"Total Bad Photos:  {total_bad} | Wrongly Accepted: {bad_accepted} ({false_acceptance_rate:.1f}%)")

--- Evaluating folder: C:\Users\VIVEK\Downloads\WhatsApp Unknown 2026-11-09 at 11.43.06 AM ---

[FALSE ACCEPTANCE] bad (20).jpeg -> Bad photo slipped through!
[FALSE ACCEPTANCE] bad(73).png -> Bad photo slipped through!
[FALSE REJECTION] good (12).png -> Failed because: Ankles are cut off
[FALSE REJECTION] good (14).png -> Failed because: Feet are cut off at the bottom edge

--- FINAL SUMMARY ---
Total Good Photos: 30 | Wrongly Rejected: 2 (6.7%)
Total Bad Photos:  68 | Wrongly Accepted: 2 (2.9%)


In [15]:
import os

# Start by pointing to your 30 tune photos!
folder_path = r"C:\Users\VIVEK\Downloads\WhatsApp Unknown 2026-11-09 at 11.43.06 AM"

total_good = 0
good_rejected = 0

total_bad = 0
bad_accepted = 0

print(f"--- Evaluating folder: {folder_path} ---\n")

for filename in os.listdir(folder_path):
    if not filename.lower().endswith((".jpg", ".jpeg", ".png")):
        continue

    full_path = os.path.join(folder_path, filename)
    
    # 1. Determine Ground Truth from filename (or CSV if you made one)
    if filename.lower().startswith("good"):
        true_status = "pass"
        total_good += 1
    else:
        true_status = "fail"
        total_bad += 1

    # 2. Run your checker (check_photo ONLY reads pixels, never the filename!)
    prediction = check_photo(full_path)
    pred_status = prediction["status"]
    pred_reason = prediction["reason"]

    # 3. Check for mistakes and count them
    if true_status == "pass" and pred_status == "fail":
        good_rejected += 1
        print(f"[FALSE REJECTION] {filename} -> Failed because: {pred_reason}")
        
    elif true_status == "fail" and pred_status == "pass":
        bad_accepted += 1
        print(f"[FALSE ACCEPTANCE] {filename} -> Bad photo slipped through!")

# 4. Calculate the 2 FitFlick Business Metrics (Requirement 4)
false_rejection_rate = (good_rejected / total_good * 100) if total_good > 0 else 0
false_acceptance_rate = (bad_accepted / total_bad * 100) if total_bad > 0 else 0

print("\n--- FINAL SUMMARY ---")
print(f"Total Good Photos: {total_good} | Wrongly Rejected: {good_rejected} ({false_rejection_rate:.1f}%)")
print(f"Total Bad Photos:  {total_bad} | Wrongly Accepted: {bad_accepted} ({false_acceptance_rate:.1f}%)")

--- Evaluating folder: C:\Users\VIVEK\Downloads\WhatsApp Unknown 2026-11-09 at 11.43.06 AM ---

[FALSE ACCEPTANCE] bad (20).jpeg -> Bad photo slipped through!
[FALSE ACCEPTANCE] bad(73).png -> Bad photo slipped through!
[FALSE REJECTION] good (12).png -> Failed because: Ankles are cut off
[FALSE REJECTION] good (14).png -> Failed because: Feet are cut off at the bottom edge

--- FINAL SUMMARY ---
Total Good Photos: 30 | Wrongly Rejected: 2 (6.7%)
Total Bad Photos:  70 | Wrongly Accepted: 2 (2.9%)
